# CS333 Part 3: Data quality checks

This notebook shows **how we found every problem in the "How we cleaned the data" table** on our page:
https://dianaqassab-stack.github.io/cs333-part3/

Each section below is one row of that table: the code that checks it, the result, and the rule we chose.
The rules themselves are written in Vega-Lite as `CLEAN` and `CLEAN_SALES` in `index.html`, so every chart uses them.

**Data:** the course file `transactions.csv` (a 10% sample of Dubai Land Department transactions), loaded straight from the course repository.

In [1]:
import pandas as pd

URL = "https://raw.githubusercontent.com/sharios/CS333/main/2026-27/Part%202/transactions.csv"

# Read every column as text first, and keep empty cells as "" (not NaN), the same way Vega-Lite reads a CSV
df = pd.read_csv(URL, dtype=str, keep_default_na=False)

# Then turn the number and date columns into real numbers and dates
for col in ["meter_sale_price", "actual_worth", "procedure_area"]:
    df[col] = pd.to_numeric(df[col])
df["instance_date"] = pd.to_datetime(df["instance_date"])

print(f"{len(df):,} rows, {len(df.columns)} columns")
df.head(3)

89,212 rows, 21 columns


,transaction_id,instance_date,trans_group_en,procedure_name_en,reg_type_en,property_type_en,property_sub_type_en,property_usage_en,rooms_en,has_parking,...,area_name_en,area_lat,area_lon,comm_num,master_project_en,nearest_metro_en,nearest_mall_en,procedure_area,actual_worth,meter_sale_price
0,1-11-2017-6842,2017-05-17,Sales,Sell,Existing Properties,Unit,Flat,Residential,Studio,1,...,Al Barsha South Fourth,25.05659,55.20799,681,Jumeirah Village Circle,Dubai Internet City,Marina Mall,34.01,425000,12496.32
1,1-11-2019-9322,2019-10-14,Sales,Sell,Existing Properties,Villa,Villa,Residential,3 B/R,0,...,Al Thanayah Fourth,25.06399,55.16458,394,Emirates Living,Nakheel Metro Station,Marina Mall,337.51,1850000,5481.32
2,1-41-2016-5693,2016-11-21,Sales,Delayed Sell,Existing Properties,Land,,Other,,0,...,Saih Shuaib 1,24.90019,54.97122,513,JABEL ALI HILLS,,,972.91,1309000,1345.45


## Check 1: Dates outside the analysis window

We look at the earliest and latest dates, and count how many rows fall outside **1 Jan 2015 to 31 Aug 2026**
(the same window as Parts 1 and 2).

In [2]:
print("Earliest date:", df.instance_date.min().date())
print("Latest date:  ", df.instance_date.max().date())

before = df.instance_date < "2015-01-01"
after  = df.instance_date >= "2026-09-01"        # September 2026 is only partly in the file
print(f"Before 2015:     {before.sum():,}")
print(f"After Aug 2026:  {after.sum():,}")
print(f"Removed total:   {(before | after).sum():,}")

w = df[~before & ~after].copy()                  # w = the rows we keep (the "window")
print(f"Kept:            {len(w):,}")

Earliest date: 1967-11-28
Latest date:   2026-09-17
Before 2015:     21,664
After Aug 2026:  526
Removed total:   22,190
Kept:            67,022


**Rule (in `CLEAN`):** keep 1 Jan 2015 to 31 Aug 2026, with dates read in UTC.

## Check 2: Are mortgage and gift values real prices?

We compare the median price per m² for each transaction type.

In [3]:
w.groupby("trans_group_en")["meter_sale_price"].agg(["count", "median"]).round(0)

,count,median
trans_group_en,,
Gifts,2891,11476.0
Mortgages,12792,7804.0
Sales,51339,15114.0


Mortgages and gifts are much lower than sales, because a mortgage value is a bank valuation and a gift isn't a sale.
**Rule (in `CLEAN_SALES`):** price charts use **Sales only**.

## Check 3: Zero or extreme prices

We look at the cheapest and most expensive sales per m², then count how many sales fall outside 1,000–100,000 AED/m².

In [4]:
sales = w[w.trans_group_en == "Sales"]
cols = ["instance_date", "property_type_en", "procedure_area", "actual_worth", "meter_sale_price", "area_name_en"]

print("Cheapest sales per m²:")
display(sales.nsmallest(5, "meter_sale_price")[cols])
print("Most expensive sales per m²:")
display(sales.nlargest(5, "meter_sale_price")[cols])

Cheapest sales per m²:


,instance_date,property_type_en,procedure_area,actual_worth,meter_sale_price,area_name_en
75350,2025-02-13,Land,3547.26,14,0.00,Wadi Al Safa 2
7744,2022-06-02,Land,109247.72,109247,1.00,Mena Jabal Ali
27533,2022-06-03,Land,31676.35,31676,1.00,Jabal Ali Industrial Third
60905,2022-06-02,Land,26362.59,26362,1.00,Mena Jabal Ali
68174,2024-03-07,Land,869597.71,1520000,1.75,Hessyan Second


Most expensive sales per m²:


,instance_date,property_type_en,procedure_area,actual_worth,meter_sale_price,area_name_en
75136,2016-11-01,Building,929.03,385000000,414410.73,Trade Center Second
25683,2022-10-07,Building,102.20,34100000,333659.49,Burj Khalifa
37338,2023-12-12,Building,929.03,305000000,328299.41,Trade Center Second
44942,2025-07-14,Villa,1099.63,180000000,163691.42,Jumeirah Second
35875,2016-01-04,Building,29.06,4690000,161390.23,Al Buteen


In [5]:
bad_price = (sales.meter_sale_price < 1000) | (sales.meter_sale_price > 100000)
units_villas = sales.property_type_en.isin(["Unit", "Villa"])

print(f"All sales outside 1,000–100,000 AED/m²:  {bad_price.sum():,} of {len(sales):,}")
print(f"Unit/villa sales outside that range:     {(bad_price & units_villas).sum():,} of {units_villas.sum():,}")

All sales outside 1,000–100,000 AED/m²:  237 of 51,339
Unit/villa sales outside that range:     75 of 46,286


**Rule (in `CLEAN_SALES`):** keep sales between **1,000 and 100,000 AED/m²**.

## Check 4: Impossible sizes

We look at the smallest units and villas. A real studio is about 25 m² or more.

In [6]:
uv = sales[units_villas]
print(f"Unit/villa sales under 15 m²: {(uv.procedure_area < 15).sum():,}")
uv.nsmallest(8, "procedure_area")[["procedure_area", "rooms_en", "actual_worth", "meter_sale_price", "area_name_en"]]

Unit/villa sales under 15 m²: 116


,procedure_area,rooms_en,actual_worth,meter_sale_price,area_name_en
3175,0.83,1 B/R,11700,14096.39,Marsa Dubai
87677,1.21,2 B/R,16465,13607.44,Marsa Dubai
59485,1.23,2 B/R,16781,13643.09,Marsa Dubai
49667,1.56,2 B/R,58810,37698.72,Marsa Dubai
14737,2.17,3 B/R,46260,21317.97,Al Wasl
41643,2.18,,22056,10117.43,Al Raffa
73084,2.68,Office,142843,53299.87,Business Bay
44918,3.18,3 B/R,32904,10347.40,Marsa Dubai


In [7]:
too_small = sales.procedure_area < 15
print(f"All sales under 15 m²:        {too_small.sum():,}")
print(f"Removed by price OR size:     {(bad_price | too_small).sum():,}  ({(bad_price | too_small).mean():.1%} of sales)")
print(f"Clean sales left:             {(~(bad_price | too_small)).sum():,}")

All sales under 15 m²:        122
Removed by price OR size:     356  (0.7% of sales)
Clean sales left:             50,983


**Rule (in `CLEAN_SALES`):** keep properties of **at least 15 m²**. Together with the price rule, this leaves the clean sales every price chart uses.

## Check 5: Missing number of rooms

In [8]:
no_rooms = w[w.rooms_en.str.strip() == ""]
print(f"Rows with no room count: {len(no_rooms):,}")
no_rooms.property_type_en.value_counts()

Rows with no room count: 12,766


property_type_en
Land        6934
Villa       4112
Unit         869
Building     851
Name: count, dtype: int64

Most are land plots (which have no rooms), but thousands of villas and units are missing it too.
**Rule (in `CLEAN`):** label them **"Not stated"** (field `rooms_clean`) instead of dropping real sales.

## Check 6: Nearest metro station: missing values and spelling mistakes

In [9]:
print(f"Rows with no station: {(w.nearest_metro_en.str.strip() == '').sum():,}")

typos = ["Buj Khalifa Dubai Mall Metro Station", "Jumeirah Beach Resdency", "Palm Deira Metro Stations"]
print(f"Rows with a misspelt station: {w.nearest_metro_en.isin(typos).sum():,}")
w[w.nearest_metro_en.isin(typos)].nearest_metro_en.value_counts()

Rows with no station: 24,990
Rows with a misspelt station: 5,802


nearest_metro_en
Buj Khalifa Dubai Mall Metro Station    5012
Jumeirah Beach Resdency                  644
Palm Deira Metro Stations                146
Name: count, dtype: int64

In [10]:
# All station names, so you can see the tram stops (e.g. Harbour Tower, Mina Seyahi) mixed in with metro stations
sorted(w.nearest_metro_en.unique())

['',
 'ADCB Metro Station',
 'Abu Baker Al Siddique Metro Station',
 'Abu Hail Metro Station',
 'Airport Free Zone',
 'Airport Terminal 1 Metro Station',
 'Al Fahidi Metro Station',
 'Al Ghubaiba Metro Station',
 'Al Jadaf Metro Station',
 'Al Jafiliya Metro Station',
 'Al Nahda Metro Station',
 'Al Qiyadah Metro Station',
 'Al Qusais Metro Station',
 'Al Ras Metro Station',
 'Al Rigga Metro Station',
 'Al Sufouh',
 'Baniyas Square Metro Station',
 'Buj Khalifa Dubai Mall Metro Station',
 'Burjuman Metro Station',
 'Business Bay Metro Station',
 'Creek Metro Station',
 'DANUBE Metro Station',
 'Damac Properties',
 'Deira City Centre',
 'Dubai Internet City',
 'Dubai Marina',
 'ENERGY Metro Station',
 'Emirates Metro Station',
 'Emirates Towers Metro Station',
 'Etisalat Metro Station',
 'Financial Centre',
 'First Abu Dhabi Bank Metro Station',
 'GGICO Metro Station',
 'Harbour Tower',
 'Healthcare City Metro Station',
 'Ibn Battuta Metro Station',
 'Jumeirah Beach Resdency',
 'Jumeira

**Rule (in `CLEAN`):** fix the three spellings and label empty ones **"No station listed"** (field `metro_clean`).

## Check 7: A label written in Arabic

In [11]:
w.property_usage_en.value_counts()

property_usage_en
Residential                 56720
Commercial                   5542
Other                        2162
Hospitality                  1907
أخرى                          293
Multi-Use                     207
Industrial                    163
Agricultural                   17
Storage                        10
Residential / Commercial        1
Name: count, dtype: int64

**Rule (in `CLEAN`):** "أخرى" means "Other", so it is merged into "Other" (field `usage_clean`).

## Check 8: Missing coordinates

In [12]:
w[w.area_lat.str.strip() == ""][["area_name_en", "comm_num", "trans_group_en"]]

,area_name_en,comm_num,trans_group_en
7134,Al Maha,845,Sales
50227,Al Layan1,987,Sales


**Rule:** keep these rows. They only drop out of point maps.

## Check 9: Days with no transactions

We list every calendar day in the window and check which ones have no transactions at all.

In [13]:
all_days = pd.date_range("2015-01-01", "2026-08-31")
days_with_data = set(w.instance_date.dt.normalize())
empty = [d for d in all_days if d not in days_with_data]
empty_weekdays = [d for d in empty if d.dayofweek < 5]      # Monday–Friday

print(f"Days in window: {len(all_days):,}")
print(f"Days with no transactions: {len(empty):,}")
print(f"  of which Monday–Friday: {len(empty_weekdays):,}")
pd.Series([d.year for d in empty_weekdays]).value_counts().sort_index().rename("empty weekdays per year")

Days in window: 4,261
Days with no transactions: 1,545
  of which Monday–Friday: 327


2015    59
2016    64
2017    55
2018    60
2019    40
2020    18
2021    14
2022     5
2023     4
2024     2
2025     2
2026     4
Name: empty weekdays per year, dtype: int64

Most empty days are weekends, and the empty weekdays are mostly in 2015–2019, when volumes were low in this 10% sample.
**Rule:** remove nothing. Time charts group by month so empty days don't look like drops.

## Check 10: Is price per m² consistent with total price ÷ area?

If it is, we can use `meter_sale_price` directly in every chart.

In [14]:
calc = w.actual_worth / w.procedure_area
gap = (calc - w.meter_sale_price).abs()
ok = gap <= 0.01 * w.meter_sale_price           # within 1%
print(f"Rows where meter_sale_price = actual_worth ÷ procedure_area (within 1%): {ok.mean():.2%}")
print(f"Rows that don't match: {(~ok).sum():,}")
w[~ok][["instance_date", "property_type_en", "procedure_area", "actual_worth", "meter_sale_price"]].assign(calculated=calc[~ok])

Rows where meter_sale_price = actual_worth ÷ procedure_area (within 1%): 100.00%
Rows that don't match: 1


,instance_date,property_type_en,procedure_area,actual_worth,meter_sale_price,calculated
75350,2025-02-13,Land,3547.26,14,0.0,0.003947


The only row that doesn't match is the land plot recorded at **14 AED** in total: 14 ÷ 3,547 m² = 0.0039, which the file
rounds to 0.00. So the two always agree apart from rounding, and `meter_sale_price` is safe to use directly.
(That sale is removed anyway by the 1,000 AED/m² rule.)

## Summary: the numbers our Vega-Lite rules produce

These match the cleaning table on the page, and the counts we got by running `CLEAN` and `CLEAN_SALES` in Vega itself.

In [15]:
clean_sales = sales[~(bad_price | too_small)]
summary = pd.Series({
    "Rows in file": len(df),
    "Rows kept by CLEAN (date window)": len(w),
    "Sales in window": len(sales),
    "Clean sales kept by CLEAN_SALES": len(clean_sales),
})
summary.map("{:,}".format)

Rows in file                        89,212
Rows kept by CLEAN (date window)    67,022
Sales in window                     51,339
Clean sales kept by CLEAN_SALES     50,983
dtype: str